# README
- **Author**: RUEI-HSIANG, CHANG（張睿庠；B12303030；AI協助：Codex）
- **Created At**: 2026-11-14
- **Last Modified At**: 2026-11-14

## What does this file do?
將既有267人資料的政黨轉為1×2向量 `[藍綠光譜位置, 黨員規模]`，另保存兩兩政黨差距向量。學生明確選擇 min–max（不是均勻分位數）：`(n-100)/331310`。查不到的人數一律100；新黨2000多人以2000近似。來源沿用本次對話已查到的五筆數字，日期與口徑不一致，並非2026同日黨員總數。

無黨籍（原值「無」）也依本次剩餘類別100規則作計算佔位，不能解釋為有100名黨員。難歸類與無黨籍的光譜位置保留null；差距沿用原notebook：兩者皆無位置為0，僅一方無位置為1。規模差為正規化值絕對差。尚未選擇兩個維度合併為scalar的距離方式，也未重跑PAM。

## What does this file take?
- Source Data Sets: `data/processed/candidates_scope_g1_g6_20261110.csv`；candidate_key唯一；267人。
- Documentation: 輸入產製說明見 `data/code/apply_g6_teammate_20261110.ipynb`。
- SHA-256: `94469ef7450ec722fc16f52206d7bbbdc3933d727cae9e962232f9bb965f489d`
- 光譜規則沿用 `finding/candidate_pam_party_spectrum.ipynb`，保留原作者與原分群結果。

## What does this file output?
- `data/processed/party_vector_20261114.csv`：每個登記政黨1列，21列；編碼與來源。
- `data/processed/candidates_scope_g1_g6_party_vector_20261114.csv`：候選人資料267列，保留所有原欄並新增編碼。
- `data/processed/party_difference_vector_20261114.csv`：21×21有向配對，441列；政黨差距的1×2向量。
- Role: analysis input / lookup；Documentation: `data/processed/party_vector_20261114_documentation.ipynb`。
- 每次執行重建這三個專用輸出；核對重讀、鍵、人數、向量維度、範圍與來源hash。實際hash見執行輸出及PROJECT.md。


In [1]:
SOURCE_REL='data/processed/candidates_scope_g1_g6_20261110.csv'
SOURCE_SHA='94469ef7450ec722fc16f52206d7bbbdc3933d727cae9e962232f9bb965f489d'
from pathlib import Path
import csv, json, hashlib, math
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').exists())
SOURCE = ROOT / SOURCE_REL
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == SOURCE_SHA
with SOURCE.open(encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f); original_columns = reader.fieldnames; candidates = list(reader)
assert len(candidates) == 267 and len({r['candidate_key'] for r in candidates}) == 267
known = {
 '中國國民黨': (331410, '2025-09-10', 'eligible_voters', 'https://money.udn.com/money/story/7307/9029253', 'reported'),
 '民主進步黨': (238664, '2023-01-15', 'eligible_voters', 'https://www.dpp.org.tw/media/contents/9936', 'reported'),
 '台灣民眾黨': (32546, '2025-08-10', 'effective_members', 'https://www.cna.com.tw/news/aipl/202508100030.aspx', 'reported'),
 '新黨': (2000, '', 'website_2000_plus', 'https://np.org.tw/', 'approximate_lower_bound'),
 '時代力量': (823, '2025', 'effective_members', 'https://newpowerparty.tw/wp-content/uploads/2026/02/npp-2025-report.pdf', 'reported')}
positions = {'中國國民黨':1., '民主進步黨':0., '台灣民眾黨':.6, '新黨':.8, '勞動黨':.8,
 '時代力量':.2, '台灣基進':.2, '台聯黨':.2, '台灣綠黨':.2, '社會民主黨':.2}
parties=[]
for p in sorted({r['party_registration_raw'] for r in candidates}):
    count,date,basis,url,status=known.get(p,(100,'','user_assumption','', 'imputed_100'))
    score=(count-100)/(331410-100)
    pos=positions.get(p)
    parties.append(dict(party_registration_raw=p, party_members_used=count, party_members_date=date,
        party_members_basis=basis, party_members_status=status, party_members_source=url,
        party_spectrum_position='' if pos is None else pos, party_size_minmax=score,
        party_vector=json.dumps([pos,score],ensure_ascii=False)))
lookup={r['party_registration_raw']:r for r in parties}
extra=list(parties[0])[1:]
augmented=[dict(r, **{k:lookup[r['party_registration_raw']][k] for k in extra}) for r in candidates]
differences=[]
for a in parties:
    for b in parties:
        pa,pb=a['party_spectrum_position'],b['party_spectrum_position']
        spectrum=abs(pa-pb) if pa!='' and pb!='' else (0. if pa==pb=='' else 1.)
        size=abs(a['party_size_minmax']-b['party_size_minmax'])
        differences.append(dict(party_a=a['party_registration_raw'],party_b=b['party_registration_raw'],
            party_spectrum_difference=spectrum, party_size_difference=size,
            party_difference_vector=json.dumps([spectrum,size])))
outputs={'party_vector_20261114.csv':parties,
         'candidates_scope_g1_g6_party_vector_20261114.csv':augmented,
         'party_difference_vector_20261114.csv':differences}
for filename,rows in outputs.items():
    path=ROOT/'data/processed'/filename
    with path.open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    with path.open(encoding='utf-8-sig',newline='') as f: saved=list(csv.DictReader(f))
    assert len(saved)==len(rows) and all(saved[i]=={k:str(v) for k,v in r.items()} for i,r in enumerate(rows))
    print(filename, 'rows=',len(saved),'sha256=',hashlib.sha256(path.read_bytes()).hexdigest())
assert min(r['party_size_minmax'] for r in parties)==0 and max(r['party_size_minmax'] for r in parties)==1
assert all(len(json.loads(r['party_difference_vector']))==2 for r in differences)
assert all(0<=r['party_size_difference']<=1 and 0<=r['party_spectrum_difference']<=1 for r in differences)
for r in parties: print(r['party_registration_raw'],r['party_members_used'],r['party_vector'],r['party_members_status'])
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==SOURCE_SHA


party_vector_20261114.csv rows= 21 sha256= f57ecc5c3f01211ee526dc82c99a7eb6922bfaa9c307a1c5ca63ea29c27e795f
candidates_scope_g1_g6_party_vector_20261114.csv rows= 267 sha256= 97b96b65b9e89a27db44768f1d4e3da91889c454b8efa5b0577231e358340f18
party_difference_vector_20261114.csv rows= 441 sha256= 4d56330bc5628eac95054c12086baa388612e86421c4b6f4abf5e56bb15a22de
三勢團結促進聯盟 100 [null, 0.0] imputed_100
中國國民黨 331410 [1.0, 1.0] reported
勞動黨 100 [0.8, 0.0] imputed_100
台灣基進 100 [0.2, 0.0] imputed_100
台灣工黨 100 [null, 0.0] imputed_100
台灣民眾黨 32546 [0.6, 0.09793244997132595] reported
台灣綠黨 100 [0.2, 0.0] imputed_100
台灣麻將最大黨 100 [null, 0.0] imputed_100
台聯黨 100 [0.2, 0.0] imputed_100
司法改革黨 100 [null, 0.0] imputed_100
司法正義黨 100 [null, 0.0] imputed_100
天宙和平統一家庭黨 100 [null, 0.0] imputed_100
小民參政歐巴桑聯盟 100 [null, 0.0] imputed_100
左翼聯盟 100 [null, 0.0] imputed_100
新黨 2000 [0.8, 0.005734810298511968] approximate_lower_bound
時代力量 823 [0.2, 0.0021822462346442906] reported
民主進步黨 238664 [0.0, 0.7200627810811626] repo